# PalgharGPT — QLoRA Domain Adaptation from THREE Palghar PDFs (No RAG)

**Goal:** continue causal-language-model pretraining of `Qwen/Qwen2.5-1.5B-Instruct` on the three supplied Palghar district PDFs, using 4-bit NF4 quantization and QLoRA adapters.

**Important scientific note:** this is continued pretraining, not supervised question-answer training and not RAG. It adapts language patterns and domain vocabulary from the documents. It does **not** guarantee that the model will recall exact facts, preserve reference dates, or answer every Palghar question correctly. Evaluate factual answers against source pages before relying on the model.

**Run:** open this notebook in Google Colab, select **Runtime → Change runtime type → T4 GPU**, then use **Runtime → Run all**. The PDF upload dialog appears in Section 2. Upload all three original PDFs when prompted.

The notebook does not install, upgrade, or replace PyTorch. It installs only missing Python dependencies. All training parameters are grouped in Section 5 and Section 9.

## 1. Environment setup

This section verifies the existing runtime, installs only missing dependencies, and stops if a CUDA GPU is unavailable. The notebook uses FP16 because the Tesla T4 does not support BF16 training.

In [1]:
import importlib.util
import subprocess
import sys
import time
import os
import json
import re
import csv
import math
import random
import hashlib
import inspect
import platform
import shutil
import zipfile
import unicodedata
from pathlib import Path
from datetime import datetime, timezone

# Map import names to pip distribution names. PyTorch is deliberately excluded.
required_packages = {
    "fitz": "PyMuPDF",
    "pandas": "pandas",
    "transformers": "transformers",
    "datasets": "datasets",
    "peft": "peft",
    "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate",
}

missing_packages = [
    pip_name for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing dependencies only:", missing_packages)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing_packages
    ])
else:
    print("All required Python packages are already installed.")

# Import PyTorch only after dependency check. Never pip-install/replace torch here.
import torch
import pandas as pd
import fitz
import transformers
import datasets
import peft
import bitsandbytes as bnb
import accelerate

print("\nRuntime versions")
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", getattr(bnb, "__version__", "version unavailable"))
print("Accelerate:", accelerate.__version__)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is unavailable. In Google Colab select Runtime > Change runtime type > T4 GPU, "
        "save, reconnect, and run the notebook again. CPU-only training is not configured."
    )

gpu = torch.cuda.get_device_properties(0)
gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gib = gpu.total_memory / (1024 ** 3)
print(f"\nGPU: {gpu_name}")
print(f"GPU VRAM: {gpu_vram_gib:.2f} GiB")
print("CUDA runtime reported by PyTorch:", torch.version.cuda)
print("BF16 supported:", torch.cuda.is_bf16_supported())

if "T4" not in gpu_name.upper():
    print("NOTICE: This notebook is tuned for a Tesla T4. Review batch size and sequence length for this GPU.")


Installing missing dependencies only: ['PyMuPDF', 'bitsandbytes']


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(



Runtime versions
Python: 3.13.16
PyTorch: 2.14.1+cu130
Transformers: 5.22.1
Datasets: 4.8.5
PEFT: 0.21.2
BitsAndBytes: 0.50.2
Accelerate: 1.15.0

GPU: Tesla T4
GPU VRAM: 14.56 GiB
CUDA runtime reported by PyTorch: 13.0
BF16 supported: True


## 2. Upload all three source PDFs

Upload these three source files:

1. `20251216907590689.pdf` — District Socio-Economic Review 2025
2. `489VBcyu4MPalgharDSP1July2024.pdf` — Palghar District Strategic Plan

The matcher tolerates browser-renamed copies such as `filename (1).pdf`. It checks the PDF signature and stops if any required document is missing.

In [3]:
from google.colab import files

EXPECTED_DOCUMENTS = {
    "20251216907590689.pdf": {
        "title": "District Socio-Economic Review 2025",
        "publication_year": 2025,
        "reference_period_note": (
            "Primarily covers 2024-25; individual tables may use different reference dates. "
            "Use the date stated with each statistic; publication year is not assumed to be its reference year."
        ),
    },
    "489VBcyu4MPalgharDSP1July2024.pdf": {
        "title": "Palghar District Strategic Plan",
        "publication_year": 2024,
        "reference_period_note": (
            "Contains multiple reference periods, including FY2016-FY2022. "
            "Use the period stated in each section or table; publication year is not assumed to be its reference year."
        ),
    },
    "Text_to_PDF_Onlinenotpad.pdf": {
        "title": "Palghar District Development Research Report",
        "publication_year": 2026,
        "reference_period_note": (
            "Research reference date stated in the report: 7 November 2026. "
            "The report combines indicators from different historical reference periods, "
            "administrative publications and recommendations. Preserve the reference year "
            "stated for each statistic. Treat proposed projects as recommendations, not "
            "approved government projects, and distinguish report statements from official statistics."
        ),
    },
}

PROJECT_DIR = Path("/content/PalgharGPT")
PDF_DIR = PROJECT_DIR / "original_pdfs"
DATASET_DIR = PROJECT_DIR / "dataset"
OUTPUT_DIR = PROJECT_DIR / "output"
for directory in (PDF_DIR, DATASET_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Select all THREE PDF files in the upload dialog: the 2025 Socio-Economic Review, the July 2024 District Strategic Plan, and the newly added Palghar District Development Research Report.")
uploaded_files = files.upload()

def canonical_upload_name(filename):
    # Remove all instances of browser duplicate suffixes such as " (1) (1)" before the extension.
    name = Path(filename).name
    cleaned = re.sub(r"(?:\s*\(\d+\))+\s*(?=\.pdf$)", "", name, flags=re.IGNORECASE)
    return cleaned.casefold()

matched = {}
expected_by_normalized_name = {name.casefold(): name for name in EXPECTED_DOCUMENTS}

for uploaded_name, content in uploaded_files.items():
    normalized = canonical_upload_name(uploaded_name)
    if normalized in expected_by_normalized_name:
        expected_name = expected_by_normalized_name[normalized]
        if not content.startswith(b"%PDF"):
            raise ValueError(f"Uploaded file '{uploaded_name}' does not have a valid PDF signature.")
        destination = PDF_DIR / expected_name
        destination.write_bytes(content)
        matched[expected_name] = destination
        print(f"Accepted '{uploaded_name}' as '{expected_name}' ({len(content):,} bytes).")

missing = [name for name in EXPECTED_DOCUMENTS if name not in matched]
if missing:
    received = list(uploaded_files.keys())
    raise FileNotFoundError(
        "Required PDF upload(s) missing: " + ", ".join(missing) +
        ". Upload all exact source documents (duplicate suffixes such as '(1)' are automatically cleaned). "
        f"Files received: {received}"
    )

for expected_name, path in matched.items():
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Expected PDF is missing or empty: {path}")

print("\nAll three required source PDFs are present and ready for extraction.")

Select all THREE PDF files in the upload dialog: the 2025 Socio-Economic Review, the July 2024 District Strategic Plan, and the newly added Palghar District Development Research Report.


Saving Text_to_PDF_Onlinenotpad (1) (1).pdf to Text_to_PDF_Onlinenotpad (1) (1) (1).pdf
Saving 20251216907590689.pdf to 20251216907590689 (1).pdf
Saving 489VBcyu4MPalgharDSP1July2024.pdf to 489VBcyu4MPalgharDSP1July2024 (1).pdf
Accepted 'Text_to_PDF_Onlinenotpad (1) (1) (1).pdf' as 'Text_to_PDF_Onlinenotpad.pdf' (183,793 bytes).
Accepted '20251216907590689 (1).pdf' as '20251216907590689.pdf' (9,753,319 bytes).
Accepted '489VBcyu4MPalgharDSP1July2024 (1).pdf' as '489VBcyu4MPalgharDSP1July2024.pdf' (2,932,022 bytes).

All three required source PDFs are present and ready for extraction.


## 3. Extract all three PDFs page by page

PyMuPDF extracts embedded text only. Empty or near-empty pages are reported as possibly scanned/image-only pages; **no OCR is performed**. Every physical page gets a record, including empty pages, so page counts and extraction gaps remain auditable.

In [4]:
records = []
extraction_report = []

for source_name, metadata in EXPECTED_DOCUMENTS.items():
    pdf_path = matched[source_name]
    document = fitz.open(pdf_path)
    print(f"\nReading {source_name}: {len(document)} pages")

    for page_index in range(len(document)):
        page = document.load_page(page_index)
        raw_text = page.get_text("text", sort=True) or ""
        # Conservative normalization: preserve Marathi/English, digits, punctuation,
        # line boundaries, and table-like rows. Remove only null/control artifacts.
        raw_text = raw_text.replace("\x00", "")
        raw_text = raw_text.replace("\r\n", "\n").replace("\r", "\n")
        raw_text = unicodedata.normalize("NFC", raw_text)
        cleaned_lines = []
        for line in raw_text.split("\n"):
            line = re.sub(r"[\t ]+", " ", line).strip()
            cleaned_lines.append(line)
        extracted_text = "\n".join(cleaned_lines).strip()

        record = {
            "source_filename": source_name,
            "page_number": page_index + 1,
            "district": "Palghar",
            "document_title": metadata["title"],
            "publication_year": metadata["publication_year"],
            "reference_period_note": metadata["reference_period_note"],
            "extracted_text": extracted_text,
        }
        records.append(record)

        if len(extracted_text) < 40:
            extraction_report.append({
                "source_filename": source_name,
                "page_number": page_index + 1,
                "characters": len(extracted_text),
                "status": "empty_or_near_empty_possible_scanned_page",
            })

    document.close()

if not records:
    raise RuntimeError("PDF extraction returned no page records.")

print(f"\nPhysical pages represented: {len(records)}")
print(f"Empty/near-empty pages (<40 chars): {len(extraction_report)}")
for item in extraction_report:
    print(
        f"  {item['source_filename']} | page {item['page_number']} | "
        f"{item['characters']} chars | {item['status']}"
    )
print("\nNote: these pages may be scanned, image-only, blank, or contain little selectable text. OCR has not been run.")



Reading 20251216907590689.pdf: 263 pages

Reading 489VBcyu4MPalgharDSP1July2024.pdf: 118 pages

Reading Text_to_PDF_Onlinenotpad.pdf: 29 pages

Physical pages represented: 410
Empty/near-empty pages (<40 chars): 10
  20251216907590689.pdf | page 31 | 28 chars | empty_or_near_empty_possible_scanned_page
  20251216907590689.pdf | page 39 | 30 chars | empty_or_near_empty_possible_scanned_page
  20251216907590689.pdf | page 40 | 22 chars | empty_or_near_empty_possible_scanned_page
  20251216907590689.pdf | page 53 | 30 chars | empty_or_near_empty_possible_scanned_page
  20251216907590689.pdf | page 83 | 28 chars | empty_or_near_empty_possible_scanned_page
  20251216907590689.pdf | page 120 | 25 chars | empty_or_near_empty_possible_scanned_page
  20251216907590689.pdf | page 144 | 25 chars | empty_or_near_empty_possible_scanned_page
  20251216907590689.pdf | page 207 | 30 chars | empty_or_near_empty_possible_scanned_page
  20251216907590689.pdf | page 227 | 19 chars | empty_or_near_empty_p

## 4. Create and inspect the combined three-document master dataset

The JSONL file is the canonical page-level dataset. A CSV copy is also written for inspection. Exact duplicate extracted page texts are counted but not automatically deleted.

**Source coverage:** the combined JSONL/CSV includes page records from all three PDFs. Each record retains `source_filename`, `document_title`, `publication_year`, `reference_period_note`, and `page_number`. The new 2026 research report contains a mixture of historical statistics, report-level synthesis, and proposed projects; these distinctions are preserved in its page metadata and training headers.

In [5]:
import pandas as pd

MASTER_JSONL = DATASET_DIR / "palghar_master_dataset.jsonl"
MASTER_CSV = DATASET_DIR / "palghar_master_dataset.csv"

with MASTER_JSONL.open("w", encoding="utf-8") as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

pd.DataFrame(records).to_csv(MASTER_CSV, index=False, encoding="utf-8-sig")

total_chars = sum(len(row["extracted_text"]) for row in records)
pages_by_source = {
    source: sum(1 for row in records if row["source_filename"] == source)
    for source in EXPECTED_DOCUMENTS
}
nonempty_records = [row for row in records if row["extracted_text"].strip()]
text_hashes = [
    hashlib.sha256(row["extracted_text"].encode("utf-8")).hexdigest()
    for row in nonempty_records
]
duplicate_page_count = len(text_hashes) - len(set(text_hashes))
char_counts = [len(row["extracted_text"]) for row in records]

dataset_metadata = {
    "project": "PalgharGPT",
    "district": "Palghar",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_documents": EXPECTED_DOCUMENTS,
    "physical_pages_extracted": len(records),
    "nonempty_pages": len(nonempty_records),
    "total_extracted_characters": total_chars,
    "pages_per_source": pages_by_source,
    "empty_or_near_empty_pages": extraction_report,
    "exact_duplicate_nonempty_page_text_count": duplicate_page_count,
    "text_quality": {
        "pages_with_at_least_40_characters": sum(1 for n in char_counts if n >= 40),
        "pages_under_40_characters": sum(1 for n in char_counts if n < 40),
        "median_characters_per_page": float(pd.Series(char_counts).median()) if char_counts else 0,
        "minimum_characters_per_page": min(char_counts) if char_counts else 0,
        "maximum_characters_per_page": max(char_counts) if char_counts else 0,
        "note": "Character counts are extraction diagnostics, not a measure of factual quality or OCR accuracy.",
    },
}

(DATASET_DIR / "dataset_metadata.json").write_text(
    json.dumps(dataset_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Master dataset:", MASTER_JSONL)
print("CSV inspection copy:", MASTER_CSV)
print("Total pages:", len(records))
print("Total extracted characters:", f"{total_chars:,}")
print("Pages per source:", pages_by_source)
print("Empty/near-empty pages:", len(extraction_report))
print("Exact duplicate nonempty page texts:", duplicate_page_count)
print("\nSample records (text truncated for display):")
for sample in records[:3]:
    preview = sample["extracted_text"][:700]
    print("\n---", sample["source_filename"], "page", sample["page_number"], "---")
    print(preview if preview else "[NO EXTRACTABLE TEXT]")


Master dataset: /content/PalgharGPT/dataset/palghar_master_dataset.jsonl
CSV inspection copy: /content/PalgharGPT/dataset/palghar_master_dataset.csv
Total pages: 410
Total extracted characters: 566,738
Pages per source: {'20251216907590689.pdf': 263, '489VBcyu4MPalgharDSP1July2024.pdf': 118, 'Text_to_PDF_Onlinenotpad.pdf': 29}
Empty/near-empty pages: 10
Exact duplicate nonempty page texts: 0

Sample records (text truncated for display):

--- 20251216907590689.pdf page 1 ---
सत्यमेय जयते
महाराष्ट्र शासन
जिल्हा सामाजिक व आर्थिक
समालोचन - 2025
जिल्हा - पालघर





जजल्हा साांख्ययकी कायालय, पालघर
अिथ व साांख्यिकी सांचालनालि, महाराष्ट्र शासन

--- 20251216907590689.pdf page 2 ---
जजल्हा सामाजजक व आर्थथक समालोचन
2025
जजल्हा - पालघर





अथथ व साांख्ययकी सांचालनालय,
महाराष्ट्र शासन,
पालघर

--- 20251216907590689.pdf page 3 ---
प्रस्तावना

जिल्हा सामाजिक व आर्थथक समालोचन हे प्रकाशन अथथ व साांख्ययकी सांचालनालयामार्थत दरवषी

प्रकाजशत करण्यात येते. पालघर जिल्याची स्थापना जद. 01 ऑगस्ट 2014 रोिी झाली 

## 5. Combined three-document dataset preparation and reproducible page-level split

All non-empty extracted pages are eligible; there is no single-topic page filter. The split happens **before tokenization**, at page level, so chunks from one page cannot appear in both training and validation. The random seed and page assignments are recorded.

The model is trained on source text with a compact metadata header. Each page ends with the tokenizer EOS token, and chunks never cross page boundaries.

The train/validation split is performed across pages from all three documents. Exact duplicate page texts are grouped together across sources to reduce leakage.

In [6]:
# Reproducible dataset preparation settings.
RANDOM_SEED = 42
VALIDATION_FRACTION = 0.10
MAX_SEQUENCE_LENGTH = 512

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

eligible_pages = [
    row for row in records
    if isinstance(row["extracted_text"], str) and row["extracted_text"].strip()
]
if len(eligible_pages) < 2:
    raise ValueError(
        f"Only {len(eligible_pages)} non-empty pages were extracted. "
        "At least two are required for a page-level train/validation split. "
        "Inspect the PDFs; scanned pages may require a separate OCR workflow."
    )

# Group exact duplicate page texts so identical extracted content cannot land
# in both splits. Every group still contains whole pages; tokenization happens later.
duplicate_groups = {}
for row in eligible_pages:
    digest = hashlib.sha256(row["extracted_text"].encode("utf-8")).hexdigest()
    duplicate_groups.setdefault(digest, []).append(row)

groups = list(duplicate_groups.values())
if len(groups) < 2:
    raise ValueError(
        "All non-empty pages have identical extracted text, so a leakage-free "
        "page-level validation split is impossible."
    )

random.Random(RANDOM_SEED).shuffle(groups)
target_validation_pages = max(1, int(round(len(eligible_pages) * VALIDATION_FRACTION)))
validation_groups = []
validation_page_total = 0

# Select whole duplicate groups until the target is reached, reserving at least
# one group for training.
for group in groups[:-1]:
    if validation_page_total < target_validation_pages:
        validation_groups.append(group)
        validation_page_total += len(group)

if not validation_groups:
    validation_groups = [groups[0]]
    groups = groups[1:]

validation_group_ids = {id(group) for group in validation_groups}
validation_pages = [row for group in validation_groups for row in group]
training_pages = [
    row for group in groups if id(group) not in validation_group_ids for row in group
]

train_page_keys = {
    (row["source_filename"], row["page_number"]) for row in training_pages
}
validation_page_keys = {
    (row["source_filename"], row["page_number"]) for row in validation_pages
}
assert train_page_keys.isdisjoint(validation_page_keys), "Page-level train/validation leakage detected."

train_text_hashes = {
    hashlib.sha256(row["extracted_text"].encode("utf-8")).hexdigest()
    for row in training_pages
}
validation_text_hashes = {
    hashlib.sha256(row["extracted_text"].encode("utf-8")).hexdigest()
    for row in validation_pages
}
assert train_text_hashes.isdisjoint(validation_text_hashes), "Exact duplicate text leakage detected."

def page_to_training_text(row):
    header = (
        f"[District: {row['district']}]\\n"
        f"[Document: {row['document_title']}]\\n"
        f"[Source file: {row['source_filename']}]\\n"
        f"[Page: {row['page_number']}]\\n"
        f"[Publication year: {row['publication_year']}]\\n"
        f"[Reference-period note: {row['reference_period_note']}]\\n\\n"
    )
    return header + row["extracted_text"]

split_manifest = {
    "random_seed": RANDOM_SEED,
    "validation_fraction_requested": VALIDATION_FRACTION,
    "training_page_count": len(training_pages),
    "validation_page_count": len(validation_pages),
    "exact_duplicate_text_groups_kept_together": True,
    "training_pages": [
        {"source_filename": r["source_filename"], "page_number": r["page_number"]}
        for r in training_pages
    ],
    "validation_pages": [
        {"source_filename": r["source_filename"], "page_number": r["page_number"]}
        for r in validation_pages
    ],
}
(DATASET_DIR / "page_split_manifest.json").write_text(
    json.dumps(split_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Training pages: {len(training_pages)}")
print(f"Validation pages: {len(validation_pages)}")
print("Page-level leakage check: PASSED")
print("Exact duplicate-text leakage check: PASSED")
print("Marathi/English text is retained; no topic-based page filtering is applied.")


Training pages: 369
Validation pages: 41
Page-level leakage check: PASSED
Exact duplicate-text leakage check: PASSED
Marathi/English text is retained; no topic-based page filtering is applied.


## 6. Load Qwen2.5-1.5B-Instruct in 4-bit NF4

The base model is loaded from its pretrained checkpoint, not initialized randomly. The quantized base weights are frozen; trainable LoRA adapters are added in Section 8.

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Tesla T4: FP16 compute with 4-bit NF4 weights and nested/double quantization.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    trust_remote_code=False,
    use_fast=True,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading 4-bit pretrained model. The first run may download model files...")
try:
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=False,
        low_cpu_mem_usage=True,
    )
except TypeError as exc:
    # Some Transformers releases changed/deprecated torch_dtype in favor of dtype.
    if "torch_dtype" not in str(exc):
        raise
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        dtype=torch.float16,
        trust_remote_code=False,
        low_cpu_mem_usage=True,
    )

model.config.use_cache = False

print("\nModel loaded:", model.__class__.__name__)
print("Tokenizer vocabulary size:", len(tokenizer))
print("Tokenizer EOS token/id:", tokenizer.eos_token, tokenizer.eos_token_id)
print("Tokenizer PAD token/id:", tokenizer.pad_token, tokenizer.pad_token_id)
print("Quantization config:", model.config.quantization_config)
print("Model device map:", getattr(model, "hf_device_map", "not exposed"))
print(f"GPU allocated after load: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GiB")
print(f"GPU reserved after load: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GiB")

if not getattr(model, "is_loaded_in_4bit", False):
    raise RuntimeError("The base model does not report 4-bit loading. Stop before training.")


Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded: Qwen2ForCausalLM
Tokenizer vocabulary size: 151665
Tokenizer EOS token/id: <|im_end|> 151645
Tokenizer PAD token/id: <|endoftext|> 151643
Quantization config: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

Model device map: not exposed
GPU allocated after load: 1.07 GiB
GPU reserved after load: 1.12 GiB


## 7. Convert page text into causal-LM training chunks

Each page is tokenized independently. Its EOS token is appended before chunking; no chunk contains text from a different page. The final short chunk of each page is retained when it contains at least two tokens. This is unsupervised causal language-model training: labels are the input tokens shifted by the model internally, not invented question-answer pairs.

In [8]:
from datasets import Dataset

def pages_to_token_chunks(page_rows, split_name):
    chunks = []
    page_chunk_counts = []
    for row in page_rows:
        text = page_to_training_text(row)
        token_ids = tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
            return_attention_mask=False,
        )["input_ids"]
        if tokenizer.eos_token_id is not None:
            token_ids = token_ids + [tokenizer.eos_token_id]

        produced = 0
        for start in range(0, len(token_ids), MAX_SEQUENCE_LENGTH):
            piece = token_ids[start:start + MAX_SEQUENCE_LENGTH]
            if len(piece) < 2:
                continue
            chunks.append({
                "input_ids": piece,
                "attention_mask": [1] * len(piece),
            })
            produced += 1

        page_chunk_counts.append({
            "split": split_name,
            "source_filename": row["source_filename"],
            "page_number": row["page_number"],
            "token_count_including_eos": len(token_ids),
            "chunk_count": produced,
        })

    return chunks, page_chunk_counts

train_chunks, train_chunk_manifest = pages_to_token_chunks(training_pages, "train")
eval_chunks, eval_chunk_manifest = pages_to_token_chunks(validation_pages, "validation")

if not train_chunks:
    raise RuntimeError("No training chunks were produced. Check PDF text extraction and tokenizer output.")
if not eval_chunks:
    raise RuntimeError("No validation chunks were produced. Increase validation pages or inspect extracted text.")

train_dataset = Dataset.from_list(train_chunks)
eval_dataset = Dataset.from_list(eval_chunks)

chunk_manifest = {
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "training_chunk_count": len(train_chunks),
    "validation_chunk_count": len(eval_chunks),
    "training_page_chunks": train_chunk_manifest,
    "validation_page_chunks": eval_chunk_manifest,
}
(DATASET_DIR / "token_chunk_manifest.json").write_text(
    json.dumps(chunk_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Training chunks:", len(train_dataset))
print("Validation chunks:", len(eval_dataset))
print("Training tokens:", sum(len(x) for x in train_dataset["input_ids"]))
print("Validation tokens:", sum(len(x) for x in eval_dataset["input_ids"]))
print("Example chunk length:", len(train_dataset[0]["input_ids"]))
print(
    "\nMethod: continued causal language-model pretraining. "
    "This teaches statistical text patterns and domain vocabulary, not guaranteed factual Q&A."
)


Training chunks: 908
Validation chunks: 99
Training tokens: 367626
Validation tokens: 41245
Example chunk length: 512

Method: continued causal language-model pretraining. This teaches statistical text patterns and domain vocabulary, not guaranteed factual Q&A.


## 8. Configure QLoRA adapters

The base model is prepared for k-bit training, then LoRA is attached to the Qwen attention and MLP projection layers. Only adapter parameters should be trainable. Gradient checkpointing reduces activation memory.

In [9]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Prepare API kwargs according to the installed PEFT version.
prepare_kwargs = {"use_gradient_checkpointing": True}
prepare_signature = inspect.signature(prepare_model_for_kbit_training)
if "gradient_checkpointing_kwargs" in prepare_signature.parameters:
    prepare_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

model = prepare_model_for_kbit_training(model, **prepare_kwargs)
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percent = 100.0 * trainable_params / total_params

print(f"Total parameters: {total_params:,}")
print(f"Trainable adapter parameters: {trainable_params:,}")
print(f"Trainable percentage: {trainable_percent:.4f}%")
model.print_trainable_parameters()

if trainable_params == 0:
    raise RuntimeError("No trainable LoRA parameters found.")
if trainable_params >= total_params:
    raise RuntimeError("Unexpectedly all model parameters are trainable; base model should be frozen.")

# Enable gradient checkpointing explicitly if the installed PEFT/model combination supports it.
try:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
except TypeError:
    model.gradient_checkpointing_enable()
except AttributeError:
    print("Model does not expose gradient_checkpointing_enable; PEFT preparation was still requested.")


Total parameters: 907,081,216
Trainable adapter parameters: 18,464,768
Trainable percentage: 2.0356%
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 9. Train and evaluate

Conservative T4-oriented defaults: micro-batch size 1, gradient accumulation 8, sequence length 512, FP16, and 3 epochs. The settings are editable below. Trainer argument names and constructor options are filtered against the installed Transformers API to tolerate version differences.

If CUDA runs out of memory, lower `MAX_SEQUENCE_LENGTH` (then rerun Sections 7–9), or lower `GRADIENT_ACCUMULATION_STEPS` only if needed. Batch size is already 1.

In [10]:
# Training settings — edit these values if desired.
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.3
WARMUP_RATIO = 0.03
LOGGING_STEPS = 10
SAVE_TOTAL_LIMIT = 2
DATALOADER_NUM_WORKERS = 0

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

training_args_values = {
    "output_dir": str(OUTPUT_DIR / "checkpoints"),
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "max_grad_norm": MAX_GRAD_NORM,
    "warmup_ratio": WARMUP_RATIO,
    "logging_steps": LOGGING_STEPS,
    "save_total_limit": SAVE_TOTAL_LIMIT,
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "dataloader_num_workers": DATALOADER_NUM_WORKERS,
    "report_to": "none",
    "save_strategy": "epoch",
    "logging_strategy": "steps",
    "remove_unused_columns": True,
    "optim": "adamw_torch",
    "seed": RANDOM_SEED,
    "data_seed": RANDOM_SEED,
}

ta_signature = inspect.signature(TrainingArguments.__init__)
ta_parameters = ta_signature.parameters
if "eval_strategy" in ta_parameters:
    training_args_values["eval_strategy"] = "epoch"
elif "evaluation_strategy" in ta_parameters:
    training_args_values["evaluation_strategy"] = "epoch"
else:
    raise RuntimeError(
        "Installed Transformers TrainingArguments exposes neither eval_strategy nor evaluation_strategy. "
        "Please inspect the installed Transformers API before training."
    )

supported_training_args = {
    key: value for key, value in training_args_values.items()
    if key in ta_parameters
}
ignored_training_args = sorted(set(training_args_values) - set(supported_training_args))
if ignored_training_args:
    print("Trainer argument(s) not supported by this Transformers version and omitted:", ignored_training_args)

training_args = TrainingArguments(**supported_training_args)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": eval_dataset,
    "data_collator": data_collator,
}
trainer_signature = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer ready.")
print("Train batches per epoch:", math.ceil(len(train_dataset) / PER_DEVICE_TRAIN_BATCH_SIZE))
print("Eval batches:", math.ceil(len(eval_dataset) / PER_DEVICE_EVAL_BATCH_SIZE))
print("FP16:", getattr(training_args, "fp16", None))
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
training_started = time.time()

try:
    train_result = trainer.train()
except torch.cuda.OutOfMemoryError as exc:
    torch.cuda.empty_cache()
    raise RuntimeError(
        "CUDA out of memory during training. Reduce MAX_SEQUENCE_LENGTH (for example 384 or 256), "
        "rerun Sections 7–9, and try again. Keep micro-batch size at 1. "
        "If necessary, reduce LoRA target modules or use a smaller base model."
    ) from exc
except Exception as exc:
    raise RuntimeError(
        f"Trainer failed: {type(exc).__name__}: {exc}. "
        "Check the version/API messages above and the Colab runtime logs."
    ) from exc

training_duration_seconds = time.time() - training_started
print("\nTraining completed. Trainer-reported metrics:")
print(train_result.metrics)

print("\nRunning held-out validation evaluation...")
try:
    eval_metrics = trainer.evaluate()
except Exception as exc:
    raise RuntimeError(f"Validation evaluation failed: {type(exc).__name__}: {exc}") from exc

print("Validation metrics:", eval_metrics)
print(f"Measured wall-clock training duration: {training_duration_seconds / 60:.2f} minutes")
print(f"Peak GPU allocated: {torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GiB")
print(f"Peak GPU reserved: {torch.cuda.max_memory_reserved(0) / 1024**3:.2f} GiB")

if "eval_loss" not in eval_metrics or not math.isfinite(float(eval_metrics["eval_loss"])):
    raise RuntimeError("Validation loss is missing or non-finite; do not treat this run as a successful evaluation.")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.


Trainer argument(s) not supported by this Transformers version and omitted: ['warmup_ratio']
Trainer ready.
Train batches per epoch: 908
Eval batches: 99
FP16: True
Gradient accumulation: 8


Epoch,Training Loss,Validation Loss
1,1.379551,1.341109
2,1.164916,1.243098
3,1.019094,1.227106



Training completed. Trainer-reported metrics:
{'train_runtime': 2019.5644, 'train_samples_per_second': 1.349, 'train_steps_per_second': 0.169, 'total_flos': 8793061292832768.0, 'train_loss': 1.2397970881378442, 'epoch': 3.0}

Running held-out validation evaluation...


Training Loss,Validation Loss,Epoch
1.019094,1.227106,3


Validation metrics: {'eval_loss': 1.2271056175231934}
Measured wall-clock training duration: 33.67 minutes
Peak GPU allocated: 3.19 GiB
Peak GPU reserved: 3.66 GiB


## 10. Evaluation and generation smoke test

Training loss measures fit to training chunks. Validation loss measures next-token prediction loss on held-out pages from this same small document collection. Lower validation loss can indicate better text prediction on this split, but it does **not** prove factual accuracy, completeness, correct attribution, or reliable answers to user questions.

The prompts below are smoke tests only. Read generated answers critically and verify every factual claim against the original PDF page. A fluent answer is not evidence that it is correct.

In [11]:
# Summarize the principal training/evaluation metrics.
training_metrics = dict(getattr(train_result, "metrics", {}) or {})
evaluation_summary = {
    "training_loss": training_metrics.get("train_loss"),
    "validation_loss": eval_metrics.get("eval_loss"),
    "training_duration_seconds_wall_clock": training_duration_seconds,
    "trainer_train_runtime_seconds": training_metrics.get("train_runtime"),
    "peak_gpu_allocated_gib": torch.cuda.max_memory_allocated(0) / 1024**3,
    "peak_gpu_reserved_gib": torch.cuda.max_memory_reserved(0) / 1024**3,
    "training_page_count": len(training_pages),
    "validation_page_count": len(validation_pages),
    "training_chunk_count": len(train_dataset),
    "validation_chunk_count": len(eval_dataset),
}
print(json.dumps(evaluation_summary, indent=2, ensure_ascii=False))

model.eval()
test_prompts = [
    "Describe the major development themes discussed for Palghar district. Distinguish facts from uncertainty.",
    "What kinds of information about agriculture and irrigation are covered in the Palghar source documents?",
    "Summarize the education and healthcare topics documented for Palghar district, and mention when reference periods should be checked.",
]

generation_device = next(
    (parameter.device for parameter in model.parameters() if parameter.device.type == "cuda"),
    torch.device("cuda:0"),
)

for prompt in test_prompts:
    formatted_prompt = (
        "You are PalgharGPT, a language model adapted on documents about Palghar district, Maharashtra. "
        "Do not invent facts. If the source context is insufficient, say so.\n\n"
        f"User: {prompt}\nPalgharGPT:"
    )
    encoded = tokenizer(formatted_prompt, return_tensors="pt")
    encoded = {key: value.to(generation_device) for key, value in encoded.items()}
    with torch.no_grad():
        generated = model.generate(
            **encoded,
            max_new_tokens=160,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(
        generated[0][encoded["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )
    print("\n" + "=" * 90)
    print("PROMPT:", prompt)
    print("GENERATED:", answer.strip() or "[No generated continuation]")


{
  "training_loss": 1.2397970881378442,
  "validation_loss": 1.2271056175231934,
  "training_duration_seconds_wall_clock": 2020.0589852333069,
  "trainer_train_runtime_seconds": 2019.5644,
  "peak_gpu_allocated_gib": 3.1918458938598633,
  "peak_gpu_reserved_gib": 3.658203125,
  "training_page_count": 369,
  "validation_page_count": 41,
  "training_chunk_count": 908,
  "validation_chunk_count": 99
}

PROMPT: Describe the major development themes discussed for Palghar district. Distinguish facts from uncertainty.
GENERATED: The district has been identified as one of the fastest-growing districts in terms of its economic growth and development. It has a rich cultural heritage and offers a unique blend of coastal and tribal landscapes. The district's economy is driven by industries such as textiles, fisheries, and agriculture, with the presence of several industrial clusters. The district also has a significant tourism potential, thanks to its natural beauty and historical sites. However,

## 11. Save PalgharGPT adapter, tokenizer, manifests, and ZIP

This saves the trained LoRA adapter (not a full copy of the quantized base model), tokenizer, training configuration, metrics, and dataset/split manifests. The original base checkpoint remains `Qwen/Qwen2.5-1.5B-Instruct` and is required when reloading the adapter.

In [12]:
FINAL_MODEL_DIR = OUTPUT_DIR / "PalgharGPT_LoRA"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save PEFT adapter and tokenizer.
trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

# Persist run configuration and evaluation evidence.
training_configuration = {
    "project_name": "PalgharGPT",
    "method": "continued causal language-model pretraining with QLoRA; no RAG",
    "base_model": BASE_MODEL_ID,
    "quantization": {
        "load_in_4bit": True,
        "quant_type": "nf4",
        "double_quantization": True,
        "compute_dtype": "float16",
        "device_map": "auto",
    },
    "lora": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
        "bias": "none",
    },
    "training": {
        "epochs": NUM_TRAIN_EPOCHS,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM,
        "warmup_ratio": WARMUP_RATIO,
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "seed": RANDOM_SEED,
        "fp16": True,
        "gradient_checkpointing": True,
    },
    "runtime": {
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "peft": peft.__version__,
        "bitsandbytes": getattr(bnb, "__version__", "unknown"),
        "gpu": torch.cuda.get_device_name(0),
        "gpu_vram_gib": gpu_vram_gib,
    },
    "scientific_limitations": [
        "Continued pretraining is not supervised question-answer training.",
        "No retrieval, vector database, embeddings, or RAG are used.",
        "The model may forget exact statistics or confuse reference periods.",
        "Generated claims require source-page verification and a separate factual evaluation.",
    ],
}

(FINAL_MODEL_DIR / "training_configuration.json").write_text(
    json.dumps(training_configuration, ensure_ascii=False, indent=2), encoding="utf-8"
)
(FINAL_MODEL_DIR / "evaluation_metrics.json").write_text(
    json.dumps(evaluation_summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

# Include source/dataset metadata and the page split/chunk manifests for reproducibility.
for filename in [
    "dataset_metadata.json",
    "page_split_manifest.json",
    "token_chunk_manifest.json",
]:
    source_path = DATASET_DIR / filename
    if source_path.exists():
        shutil.copy2(source_path, FINAL_MODEL_DIR / filename)

# Record source filenames and sizes; the original PDFs remain separately in original_pdfs/.
training_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "base_model": BASE_MODEL_ID,
    "adapter_directory": str(FINAL_MODEL_DIR),
    "dataset_jsonl": str(MASTER_JSONL),
    "dataset_csv": str(MASTER_CSV),
    "source_pdfs": [
        {
            "filename": name,
            "bytes": (PDF_DIR / name).stat().st_size,
            "publication_year": EXPECTED_DOCUMENTS[name]["publication_year"],
            "document_title": EXPECTED_DOCUMENTS[name]["title"],
            "reference_period_note": EXPECTED_DOCUMENTS[name]["reference_period_note"],
        }
        for name in EXPECTED_DOCUMENTS
    ],
    "training_pages": len(training_pages),
    "validation_pages": len(validation_pages),
    "training_chunks": len(train_dataset),
    "validation_chunks": len(eval_dataset),
    "evaluation": evaluation_summary,
}
(FINAL_MODEL_DIR / "training_manifest.json").write_text(
    json.dumps(training_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

# Save Trainer state and logs when available.
try:
    trainer.save_state()
except Exception as exc:
    print("Trainer state save warning:", exc)

try:
    logs = getattr(trainer.state, "log_history", [])
    (FINAL_MODEL_DIR / "trainer_log_history.json").write_text(
        json.dumps(logs, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
    )
except Exception as exc:
    print("Trainer log save warning:", exc)

ZIP_PATH = OUTPUT_DIR / "PalgharGPT_QLoRA_No_RAG.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in FINAL_MODEL_DIR.rglob("*"):
        if file_path.is_file():
            archive.write(file_path, arcname=str(file_path.relative_to(OUTPUT_DIR)))

if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size == 0:
    raise RuntimeError("ZIP archive was not created correctly.")

print("Adapter and supporting files saved to:", FINAL_MODEL_DIR)
print("ZIP archive:", ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MiB")
print("\nFiles included:")
with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    for item in archive.namelist():
        print(" -", item)

print("\nStarting automatic download...")
files.download(str(ZIP_PATH))


Adapter and supporting files saved to: /content/PalgharGPT/output/PalgharGPT_LoRA
ZIP archive: /content/PalgharGPT/output/PalgharGPT_QLoRA_No_RAG.zip
ZIP size: 67.25 MiB

Files included:
 - PalgharGPT_LoRA/adapter_config.json
 - PalgharGPT_LoRA/tokenizer_config.json
 - PalgharGPT_LoRA/training_configuration.json
 - PalgharGPT_LoRA/token_chunk_manifest.json
 - PalgharGPT_LoRA/adapter_model.safetensors
 - PalgharGPT_LoRA/evaluation_metrics.json
 - PalgharGPT_LoRA/trainer_log_history.json
 - PalgharGPT_LoRA/chat_template.jinja
 - PalgharGPT_LoRA/training_args.bin
 - PalgharGPT_LoRA/tokenizer.json
 - PalgharGPT_LoRA/README.md
 - PalgharGPT_LoRA/training_manifest.json
 - PalgharGPT_LoRA/page_split_manifest.json
 - PalgharGPT_LoRA/dataset_metadata.json

Starting automatic download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reload PalgharGPT in a later Colab session

Upload/extract the ZIP (or mount Drive and point to the extracted adapter folder), then load the same base model in 4-bit and attach the saved adapter. This reload example assumes the required packages are installed and a CUDA GPU is active.

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_id = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_dir = "/content/PalgharGPT/output/PalgharGPT_LoRA"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
base_model = AutoModelForCausalLM.from_pretrained(
    base_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model = PeftModel.from_pretrained(base_model, adapter_dir)
model.eval()
```

**What this notebook produces**
- `dataset/palghar_master_dataset.jsonl` — one record per PDF page, including source and reference-period metadata.
- `dataset/palghar_master_dataset.csv` — inspection copy.
- Dataset metadata, extraction diagnostics, page-level split manifest, and token-chunk manifest.
- `output/PalgharGPT_LoRA/` — LoRA adapter, tokenizer, configuration, metrics, manifests, and Trainer logs.
- `output/PalgharGPT_QLoRA_No_RAG.zip` — downloadable adapter package.

**Practical cautions**
- Colab sessions are temporary. Download the ZIP and dataset files or copy them to Google Drive before disconnecting.
- The PDF text extraction is not OCR. Scanned pages with little selectable text will be listed and excluded from training if they contain no extracted text.
- Three documents are still a small corpus. Consider adding carefully sourced, current documents and running a page-cited factual benchmark before presenting PalgharGPT as a dependable public assistant.
- Do not expose the adapter as a factual authority without checking answers against original source pages, especially for statistics and dates.